# מטלה בלמידת מכונה – NLP – K-Means ממומש מאפס

> בהתאם להבהרת המרצה: אלגוריתם הלמידה המרכזי במחברת זו הוא **K-Means from scratch**.  
> אין שימוש ב־`sklearn.cluster.KMeans` או באלגוריתם למידה מוכן מספרייה.

## פרטי סטודנטים
- שם סטודנט/ית: `<שם פרטי + אות ראשונה של שם משפחה>`
- 4 ספרות אחרונות של ת.ז.: `<XXXX>`

## פרטי המטלה
- Assignment type: **ניתוח טקסט / NLP**
- Learning type: **Clustering**
- Learning algorithm implemented: **K-Means from scratch**
- Dataset: **Spam Text Message Classification**
- Dataset URL: https://www.kaggle.com/datasets/team-ai/spam-text-message-classification

## שימוש ב־AI / Chatbot

נעזרתי ב־AI לצורך בניית שלד המחברת, ניסוח הסברים, ותכנון מימוש K-Means מאפס.  
האלגוריתם עצמו נכתב ידנית במחברת ולא מופעל באמצעות מימוש מוכן.

## הסבר על הבעיה וה־Dataset

ה־Dataset מכיל הודעות SMS המסומנות כ־`ham` או `spam`.  
במטלה זו אנו מבצעים ניתוח טקסט, הופכים הודעות לווקטורים מספריים, ומפעילים **K-Means** שמומש מאפס כדי לאשכל את ההודעות לשני אשכולות.  
מכיוון ש־K-Means הוא אלגוריתם לא מונחה, התוויות אינן משמשות באימון אלא רק בשלב הערכת האיכות.

In [ ]:
import os, re, random, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# חלק 1 – טעינת Dataset והצגת Train/Test

המחברת טוענת את ה־Dataset, מבצעת חלוקה קבועה ל־train/test ומציגה 5 שורות ראשונות מכל חלק.

In [ ]:
def load_spam_dataset():
    possible_files = [
        "spam.csv", "SMSSpamCollection.csv", "SMSSpamCollection",
        "/content/spam.csv", "/content/SMSSpamCollection.csv", "/content/SMSSpamCollection"
    ]
    for path in possible_files:
        if os.path.exists(path):
            print("Loading local dataset from:", path)
            try:
                return pd.read_csv(path, encoding="latin-1")
            except Exception:
                return pd.read_csv(path, sep="\t", header=None, names=["label", "text"], encoding="latin-1")

    try:
        import kagglehub
        dataset_path = kagglehub.dataset_download("team-ai/spam-text-message-classification")
        print("Downloaded dataset path:", dataset_path)
        for root, dirs, files in os.walk(dataset_path):
            for f in files:
                if f.lower().endswith((".csv", ".txt")):
                    full_path = os.path.join(root, f)
                    print("Found file:", full_path)
                    try:
                        return pd.read_csv(full_path, encoding="latin-1")
                    except Exception:
                        return pd.read_csv(full_path, sep="\t", header=None, names=["label", "text"], encoding="latin-1")
    except Exception as e:
        print("KaggleHub download failed:", e)

    raise FileNotFoundError("לא נמצא Dataset. העלו ל-Colab קובץ spam.csv או הגדירו KaggleHub.")

raw_df = load_spam_dataset()
raw_df.head()

In [ ]:
def normalize_columns(df):
    df = df.copy().dropna(axis=1, how="all")
    cols_lower = [str(c).lower() for c in df.columns]

    if "v1" in cols_lower and "v2" in cols_lower:
        label_col = df.columns[cols_lower.index("v1")]
        text_col = df.columns[cols_lower.index("v2")]
    elif "label" in cols_lower and "text" in cols_lower:
        label_col = df.columns[cols_lower.index("label")]
        text_col = df.columns[cols_lower.index("text")]
    elif len(df.columns) >= 2:
        label_col = df.columns[0]
        text_col = df.columns[1]
    else:
        raise ValueError("לא הצלחתי לזהות עמודות label/text.")

    out = df[[label_col, text_col]].rename(columns={label_col: "label", text_col: "text"})
    out["label"] = out["label"].astype(str).str.lower().str.strip()
    out["text"] = out["text"].astype(str)
    out = out[out["label"].isin(["ham", "spam"])].dropna().reset_index(drop=True)
    out["y"] = out["label"].map({"ham": 0, "spam": 1})
    return out

df = normalize_columns(raw_df)
print(df.shape)
print(df["label"].value_counts())
df.head()

In [ ]:
def train_test_split_from_scratch(dataframe, test_size=0.2, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(dataframe))
    rng.shuffle(indices)

    test_count = int(len(indices) * test_size)
    test_idx = indices[:test_count]
    train_idx = indices[test_count:]

    return dataframe.iloc[train_idx].reset_index(drop=True), dataframe.iloc[test_idx].reset_index(drop=True)

train_df, test_df = train_test_split_from_scratch(df, test_size=0.2, seed=RANDOM_STATE)

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)

print("5 השורות הראשונות של train:")
display(train_df.head())

print("5 השורות הראשונות של test:")
display(test_df.head())

# חלק 2 – Feature Engineering

כדי להפעיל K-Means על טקסט, צריך להפוך את הטקסט לווקטורים מספריים.  
מימשנו ידנית:
- ניקוי טקסט
- Tokenization
- n-grams
- Bag of Words
- TF-IDF
- `min_df`
- `max_features`

In [ ]:
STOPWORDS = set('''
a an the and or but if to of in on for with is are was were be been being this that these those
i you he she it we they me my your his her our their at by from as not no yes do does did
'''.split())

def clean_text(text, remove_stopwords=False, keep_numbers=False):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"£|\$|€", " MONEY ", text)
    text = re.sub(r"\d+", " NUMBER " if keep_numbers else " ", text)
    text = re.sub(r"[^a-zA-Z_ ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in text.split() if len(t) > 1]
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens

def make_ngrams(tokens, ngram_range=(1,1)):
    result = []
    for n in range(ngram_range[0], ngram_range[1] + 1):
        if n == 1:
            result.extend(tokens)
        else:
            result.extend(["_".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)])
    return result

def build_vocabulary(texts, ngram_range=(1,1), min_df=1, max_features=None, remove_stopwords=False, keep_numbers=False):
    doc_freq = Counter()
    tokenized_docs = []
    for text in texts:
        terms = make_ngrams(clean_text(text, remove_stopwords, keep_numbers), ngram_range)
        tokenized_docs.append(terms)
        doc_freq.update(set(terms))

    terms = [term for term, c in doc_freq.items() if c >= min_df]
    terms = sorted(terms, key=lambda t: doc_freq[t], reverse=True)
    if max_features:
        terms = terms[:max_features]
    vocab = {term: i for i, term in enumerate(terms)}
    return vocab, tokenized_docs

def transform_bow(texts, vocab, ngram_range=(1,1), remove_stopwords=False, keep_numbers=False):
    X = np.zeros((len(texts), len(vocab)), dtype=float)
    for row, text in enumerate(texts):
        terms = make_ngrams(clean_text(text, remove_stopwords, keep_numbers), ngram_range)
        counts = Counter(terms)
        for term, count in counts.items():
            if term in vocab:
                X[row, vocab[term]] = count
    return X

def fit_vectorizer_from_scratch(train_texts, representation="tfidf", ngram_range=(1,1), min_df=1, max_features=1000, remove_stopwords=False, keep_numbers=False):
    vocab, tokenized_docs = build_vocabulary(train_texts, ngram_range, min_df, max_features, remove_stopwords, keep_numbers)
    n_docs = len(train_texts)
    df_counts = np.zeros(len(vocab), dtype=float)
    for terms in tokenized_docs:
        for term in set(terms):
            if term in vocab:
                df_counts[vocab[term]] += 1
    idf = np.log((1 + n_docs) / (1 + df_counts)) + 1
    return {
        "vocab": vocab, "idf": idf, "representation": representation,
        "ngram_range": ngram_range, "min_df": min_df, "max_features": max_features,
        "remove_stopwords": remove_stopwords, "keep_numbers": keep_numbers
    }

def transform_texts_from_scratch(texts, vectorizer):
    bow = transform_bow(
        texts, vectorizer["vocab"], vectorizer["ngram_range"],
        vectorizer["remove_stopwords"], vectorizer["keep_numbers"]
    )
    if vectorizer["representation"] == "bow":
        return bow
    doc_lengths = bow.sum(axis=1, keepdims=True)
    doc_lengths[doc_lengths == 0] = 1.0
    tf = bow / doc_lengths
    return tf * vectorizer["idf"]

In [ ]:
demo_vectorizer = fit_vectorizer_from_scratch(
    train_df["text"].values, representation="tfidf", ngram_range=(1,2),
    min_df=2, max_features=50, remove_stopwords=True, keep_numbers=True
)
demo_train_X = transform_texts_from_scratch(train_df["text"].head(3).values, demo_vectorizer)
demo_test_X = transform_texts_from_scratch(test_df["text"].head(3).values, demo_vectorizer)

print("Vocabulary size:", len(demo_vectorizer["vocab"]))
print("Train demo matrix:", demo_train_X.shape)
print("Test demo matrix:", demo_test_X.shape)

display(pd.DataFrame({
    "original_train_text": train_df["text"].head(3),
    "tokens_after_cleaning": [clean_text(t, True, True) for t in train_df["text"].head(3)]
}))
display(pd.DataFrame({
    "original_test_text": test_df["text"].head(3),
    "tokens_after_cleaning": [clean_text(t, True, True) for t in test_df["text"].head(3)]
}))

# חלק 3 – K-Means from scratch

K-Means ממומש כאן ידנית:
1. אתחול מרכזים.
2. חישוב מרחקים.
3. שיוך כל דוגמה לאשכול הקרוב.
4. עדכון מרכזים.
5. עצירה לאחר התכנסות או לאחר `max_iter`.

המימוש כולל:
- `fit`
- `predict`
- `random init`
- `kmeans++`
- `euclidean distance`
- `cosine distance`

In [ ]:
class KMeansFromScratch:
    def __init__(self, n_clusters=2, max_iter=100, tol=1e-4, init="kmeans++", distance="cosine", random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.init = init
        self.distance = distance
        self.random_state = random_state
        self.centroids_ = None
        self.labels_ = None
        self.inertia_ = None
        self.n_iter_ = 0

    def _normalize_rows(self, X):
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return X / norms

    def _distance_matrix(self, X, centroids):
        if self.distance == "euclidean":
            X_sq = np.sum(X ** 2, axis=1, keepdims=True)
            C_sq = np.sum(centroids ** 2, axis=1)
            return np.maximum(X_sq + C_sq - 2 * X @ centroids.T, 0)
        if self.distance == "cosine":
            return 1 - self._normalize_rows(X) @ self._normalize_rows(centroids).T
        raise ValueError("distance must be euclidean or cosine")

    def _init_random(self, X, rng):
        indices = rng.choice(X.shape[0], size=self.n_clusters, replace=False)
        return X[indices].copy()

    def _init_kmeans_plus_plus(self, X, rng):
        n_samples = X.shape[0]
        centroids = [X[rng.integers(0, n_samples)]]
        for _ in range(1, self.n_clusters):
            C = np.array(centroids)
            min_distances = np.min(self._distance_matrix(X, C), axis=1)
            total = np.sum(min_distances)
            if total == 0:
                next_idx = rng.integers(0, n_samples)
            else:
                probs = min_distances / total
                next_idx = rng.choice(n_samples, p=probs)
            centroids.append(X[next_idx])
        return np.array(centroids)

    def _initialize_centroids(self, X):
        rng = np.random.default_rng(self.random_state)
        if self.init == "random":
            return self._init_random(X, rng)
        if self.init == "kmeans++":
            return self._init_kmeans_plus_plus(X, rng)
        raise ValueError("init must be random or kmeans++")

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        centroids = self._initialize_centroids(X)

        for iteration in range(self.max_iter):
            distances = self._distance_matrix(X, centroids)
            labels = np.argmin(distances, axis=1)

            new_centroids = np.zeros_like(centroids)
            for k in range(self.n_clusters):
                cluster_points = X[labels == k]
                if len(cluster_points) == 0:
                    rng = np.random.default_rng(self.random_state + iteration + k)
                    new_centroids[k] = X[rng.integers(0, X.shape[0])]
                else:
                    new_centroids[k] = cluster_points.mean(axis=0)

            shift = np.linalg.norm(new_centroids - centroids)
            centroids = new_centroids
            if shift < self.tol:
                self.n_iter_ = iteration + 1
                break
        else:
            self.n_iter_ = self.max_iter

        self.centroids_ = centroids
        self.labels_ = np.argmin(self._distance_matrix(X, centroids), axis=1)
        self.inertia_ = np.sum(np.min(self._distance_matrix(X, centroids), axis=1))
        return self

    def predict(self, X):
        if self.centroids_ is None:
            raise ValueError("Model was not fitted yet.")
        return np.argmin(self._distance_matrix(np.asarray(X, dtype=float), self.centroids_), axis=1)

    def fit_predict(self, X):
        self.fit(X)
        return self.labels_

# פונקציות הערכה

K-Means מחזיר אשכולות 0/1 ולא תוויות ham/spam.  
לכן אחרי האימון נמפה כל אשכול למחלקה הנפוצה בו, ואז נחשב מדדים.

In [ ]:
id_to_label = {0: "ham", 1: "spam"}

def map_clusters_to_labels(cluster_labels, true_labels):
    mapping = {}
    for cluster in np.unique(cluster_labels):
        true_in_cluster = true_labels[cluster_labels == cluster]
        mapping[cluster] = Counter(true_in_cluster).most_common(1)[0][0] if len(true_in_cluster) else 0
    return mapping

def apply_cluster_mapping(cluster_labels, mapping):
    return np.array([mapping.get(c, 0) for c in cluster_labels])

def confusion_matrix_from_scratch(y_true, y_pred):
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm

def precision_recall_f1_binary(y_true, y_pred, positive_label=1):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((y_true == positive_label) & (y_pred == positive_label))
    fp = np.sum((y_true != positive_label) & (y_pred == positive_label))
    fn = np.sum((y_true == positive_label) & (y_pred != positive_label))
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return precision, recall, f1

def macro_f1_from_scratch(y_true, y_pred):
    return np.mean([precision_recall_f1_binary(y_true, y_pred, label)[2] for label in [0, 1]])

# בונוסים – Grid Search + 5-Fold Cross Validation

נבצע ניסויים על Feature Engineering ו־Hyperparameters:
- representation: bow/tfidf
- ngram_range
- min_df
- max_features
- remove_stopwords
- keep_numbers
- init
- distance
- max_iter

In [ ]:
def kfold_indices_from_scratch(n_samples, n_splits=5, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.arange(n_samples)
    rng.shuffle(indices)
    folds = np.array_split(indices, n_splits)
    for i in range(n_splits):
        val_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(n_splits) if j != i])
        yield train_idx, val_idx

def parameter_grid(grid):
    keys = list(grid.keys())
    values = [grid[k] for k in keys]
    def rec(i, current):
        if i == len(keys):
            yield dict(current)
        else:
            for v in values[i]:
                current[keys[i]] = v
                yield from rec(i + 1, current)
    yield from rec(0, {})

def run_single_experiment(train_texts, train_y, val_texts, val_y, params):
    vectorizer = fit_vectorizer_from_scratch(
        train_texts, params["representation"], params["ngram_range"],
        params["min_df"], params["max_features"], params["remove_stopwords"], params["keep_numbers"]
    )
    X_train = transform_texts_from_scratch(train_texts, vectorizer)
    X_val = transform_texts_from_scratch(val_texts, vectorizer)

    model = KMeansFromScratch(
        n_clusters=2, max_iter=params["max_iter"], init=params["init"],
        distance=params["distance"], random_state=RANDOM_STATE
    )
    train_clusters = model.fit_predict(X_train)
    mapping = map_clusters_to_labels(train_clusters, train_y)

    val_clusters = model.predict(X_val)
    val_pred = apply_cluster_mapping(val_clusters, mapping)

    precision, recall, f1_spam = precision_recall_f1_binary(val_y, val_pred, 1)
    return {
        "f1_spam": f1_spam,
        "macro_f1": macro_f1_from_scratch(val_y, val_pred),
        "precision_spam": precision,
        "recall_spam": recall,
        "inertia": model.inertia_,
        "n_iter": model.n_iter_
    }

In [ ]:
grid = {
    "representation": ["tfidf", "bow"],
    "ngram_range": [(1,1), (1,2)],
    "min_df": [1, 2],
    "max_features": [300, 800],
    "remove_stopwords": [False, True],
    "keep_numbers": [False, True],
    "init": ["random", "kmeans++"],
    "distance": ["cosine", "euclidean"],
    "max_iter": [50, 100]
}

all_params = list(parameter_grid(grid))
print("Number of permutations:", len(all_params))

In [ ]:
def run_grid_search_cv(train_df, max_permutations=None):
    texts, y = train_df["text"].values, train_df["y"].values
    params_to_run = all_params if max_permutations is None else all_params[:max_permutations]
    rows = []

    for exp_id, params in enumerate(params_to_run, 1):
        fold_scores = []
        for tr_idx, val_idx in kfold_indices_from_scratch(len(train_df), 5, RANDOM_STATE):
            score = run_single_experiment(
                texts[tr_idx], y[tr_idx],
                texts[val_idx], y[val_idx],
                params
            )
            fold_scores.append(score)

        row = dict(params)
        row["mean_f1_spam"] = np.mean([s["f1_spam"] for s in fold_scores])
        row["std_f1_spam"] = np.std([s["f1_spam"] for s in fold_scores])
        row["mean_macro_f1"] = np.mean([s["macro_f1"] for s in fold_scores])
        row["mean_precision_spam"] = np.mean([s["precision_spam"] for s in fold_scores])
        row["mean_recall_spam"] = np.mean([s["recall_spam"] for s in fold_scores])
        row["mean_inertia"] = np.mean([s["inertia"] for s in fold_scores])
        row["mean_n_iter"] = np.mean([s["n_iter"] for s in fold_scores])
        rows.append(row)

        if exp_id % 10 == 0:
            print(f"Finished {exp_id}/{len(params_to_run)} experiments")

    return pd.DataFrame(rows).sort_values("mean_f1_spam", ascending=False).reset_index(drop=True)

# להרצה מהירה אפשר לשים max_permutations=40.
# להגשה מלאה עדיף להשאיר None.
results_df = run_grid_search_cv(train_df, max_permutations=None)
display(results_df.head(20))

best_params = results_df.iloc[0].to_dict()
print("Best parameters:")
for k, v in best_params.items():
    print(k, "=", v)

# בונוס – טיפול ב־Imbalanced Data

ב־Dataset יש הרבה יותר `ham` מאשר `spam`.  
לכן נבצע Oversampling למחלקת המיעוט ב־train בלבד.

In [ ]:
def oversample_minority_from_scratch(train_df, label_col="label", seed=42):
    rng = np.random.default_rng(seed)
    counts = train_df[label_col].value_counts()
    majority_label, minority_label = counts.idxmax(), counts.idxmin()
    majority_df = train_df[train_df[label_col] == majority_label]
    minority_df = train_df[train_df[label_col] == minority_label]
    needed = len(majority_df) - len(minority_df)
    sampled_indices = rng.choice(minority_df.index.values, size=needed, replace=True)
    sampled_df = train_df.loc[sampled_indices]
    balanced = pd.concat([majority_df, minority_df, sampled_df], axis=0)
    return balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

balanced_train_df = oversample_minority_from_scratch(train_df)

print("Original train distribution:")
print(train_df["label"].value_counts())
print("\nBalanced train distribution:")
print(balanced_train_df["label"].value_counts())

# אימון סופי על כל ה־Train ובדיקה על Test

In [ ]:
def train_final_model(train_df_to_use, params):
    vectorizer = fit_vectorizer_from_scratch(
        train_df_to_use["text"].values,
        representation=params["representation"],
        ngram_range=params["ngram_range"],
        min_df=int(params["min_df"]),
        max_features=int(params["max_features"]),
        remove_stopwords=bool(params["remove_stopwords"]),
        keep_numbers=bool(params["keep_numbers"])
    )
    X_train = transform_texts_from_scratch(train_df_to_use["text"].values, vectorizer)
    y_train = train_df_to_use["y"].values

    model = KMeansFromScratch(
        n_clusters=2,
        max_iter=int(params["max_iter"]),
        init=params["init"],
        distance=params["distance"],
        random_state=RANDOM_STATE
    )
    train_clusters = model.fit_predict(X_train)
    mapping = map_clusters_to_labels(train_clusters, y_train)
    return model, vectorizer, mapping

final_model, final_vectorizer, final_mapping = train_final_model(balanced_train_df, best_params)

print("Final cluster mapping:", final_mapping)
print("Final inertia:", final_model.inertia_)
print("Final iterations:", final_model.n_iter_)
print("Vocabulary size:", len(final_vectorizer["vocab"]))

In [ ]:
# הצגת 2-3 דוגמאות דרך feature engineering
sample_examples = test_df.head(3).copy()
sample_examples["tokens"] = sample_examples["text"].apply(
    lambda t: clean_text(t, bool(best_params["remove_stopwords"]), bool(best_params["keep_numbers"]))
)
sample_examples["ngrams"] = sample_examples["tokens"].apply(lambda toks: make_ngrams(toks, best_params["ngram_range"]))
display(sample_examples[["label", "text", "tokens", "ngrams"]])

In [ ]:
def predict_final(texts, model, vectorizer, mapping):
    X = transform_texts_from_scratch(texts, vectorizer)
    clusters = model.predict(X)
    y_pred = apply_cluster_mapping(clusters, mapping)
    return clusters, y_pred

test_clusters, test_pred = predict_final(test_df["text"].values, final_model, final_vectorizer, final_mapping)

prediction_preview = test_df[["text", "label", "y"]].head(5).copy()
prediction_preview["cluster"] = test_clusters[:5]
prediction_preview["predicted_y"] = test_pred[:5]
prediction_preview["predicted_label"] = prediction_preview["predicted_y"].map(id_to_label)
display(prediction_preview)

In [ ]:
precision, recall, f1_spam = precision_recall_f1_binary(test_df["y"].values, test_pred, 1)
macro_f1 = macro_f1_from_scratch(test_df["y"].values, test_pred)
cm = confusion_matrix_from_scratch(test_df["y"].values, test_pred)

print("Test precision for spam:", precision)
print("Test recall for spam:", recall)
print("Test F1 for spam:", f1_spam)
print("Test macro F1:", macro_f1)
print("\nConfusion Matrix rows=true [ham, spam], cols=pred [ham, spam]:")
print(cm)

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks([0,1], ["ham", "spam"])
plt.yticks([0,1], ["ham", "spam"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.colorbar()
plt.show()

# בונוס – Explainability

נסביר את האשכולות לפי המילים בעלות הערכים הגבוהים ביותר במרכזי האשכולות.

In [ ]:
def explain_clusters_top_terms(model, vectorizer, mapping, top_n=20):
    inv_vocab = {idx: term for term, idx in vectorizer["vocab"].items()}
    rows = []
    for cluster_id in range(model.n_clusters):
        center = model.centroids_[cluster_id]
        top_indices = np.argsort(center)[::-1][:top_n]
        predicted_label = id_to_label.get(mapping.get(cluster_id), "unknown")
        for rank, idx in enumerate(top_indices, 1):
            rows.append({
                "cluster": cluster_id,
                "mapped_label": predicted_label,
                "rank": rank,
                "term": inv_vocab[idx],
                "center_value": center[idx]
            })
    return pd.DataFrame(rows)

explain_df = explain_clusters_top_terms(final_model, final_vectorizer, final_mapping, top_n=20)
display(explain_df)

# סיכום

במחברת זו בוצעו:
1. טעינת Dataset והצגת train/test.
2. NLP Feature Engineering ממומש ידנית.
3. מימוש K-Means מאפס.
4. פונקציות fit ו־predict.
5. Grid Search ו־5-Fold Cross Validation.
6. ניסויי Feature Engineering ו־Hyperparameters.
7. Oversampling ל־Data Imbalanced.
8. חיזוי 5 הדוגמאות הראשונות ב־test.
9. חישוב F1 עבור spam, Precision, Recall, Macro-F1 ו־Confusion Matrix.
10. Explainability לפי מרכזי האשכולות.

הנקודה החשובה להצגה בסרטון: **K-Means לא השתמש בתוויות בזמן האימון; התוויות שימשו רק להערכת האיכות אחרי האימון.**